In [12]:
import pandas as pd

train = pd.read_csv("../data/train_features.csv")
X = train.drop(columns = ["price", "id"]).astype(float)
y = train["price"]
n = len(train)
bins = []

for i in range(5):
    bins.append([round(i * n / 5), round((i + 1) * n / 5)])

for i in range(5):
    print(f"Grupo {i}, de {bins[i][0]} a {bins[i][1]}, con longitud {bins[i][1] - bins[i][0]}")

assert sum([bins[i][1] - bins[i][0] for i in range(5)]) == n

Grupo 0, de 0 a 2549, con longitud 2549
Grupo 1, de 2549 a 5099, con longitud 2550
Grupo 2, de 5099 a 7648, con longitud 2549
Grupo 3, de 7648 a 10198, con longitud 2550
Grupo 4, de 10198 a 12747, con longitud 2549


In [45]:
import numpy as np

rmses = []

for j in range(5):

    X_va = X.iloc[bins[j][0] : bins[j][1]]
    X_tr = pd.concat([X.iloc[bins[i][0] : bins[i][1]] if i != j else None for i in range(5)])

    X_tr_mean = X_tr.mean()
    X_tr_std = X_tr.std()

    assert len(X_tr_mean) == 132

    X_va = ((X_va - X_tr_mean) / X_tr_std).to_numpy()
    X_tr = ((X_tr - X_tr_mean) / X_tr_std).to_numpy()
    y_va = y.iloc[bins[j][0] : bins[j][1]].to_numpy()
    y_tr = pd.concat([y.iloc[bins[i][0] : bins[i][1]] if i != j else None for i in range(5)]).to_numpy()
    w = np.zeros(X_tr.shape[1])
    b = 0
    lr = 0.1
    n = len(X_tr)

    assert len(X_va) + len(X_tr) == len(X)

    assert train["id"].iloc[bins[j][0] : bins[j][1]].isin(pd.concat([train["id"].iloc[bins[i][0] : bins[i][1]] if i != j else None for i in range(5)]).to_numpy()).sum() == 0

    for _ in range(1000):
        y_hat = X_tr @ w + b
        error = y_tr - y_hat
        dw = - 2 / n * X_tr.T @ error
        db = - 2 * np.mean(error)
        w -= lr * dw
        b -= lr * db

    y_hat = X_va @ w + b
    error = y_va - y_hat
    rmse = np.mean(error ** 2) ** 0.5
    rmses.append(rmse)

print(rmses)
print(np.mean(rmses))
print(np.std(rmses))

[np.float64(115.39567745009653), np.float64(115.37241415013311), np.float64(123.88088827711444), np.float64(107.77979839411371), np.float64(133.53880503452322)]
119.19351666119618
8.798364881424032
